## **1. Data Load**

In [2]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

df = pd.read_csv("../1_data/processed/02_preprocessed.csv")
print("Data loaded successfully!")
print(f"Shape: {df.shape}")

Data loaded successfully!
Shape: (22980, 17)


## **2. Initialize VADER and Sample Test**

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# Test on a sample
sample = df['cleaned_review_BE'].iloc[0]
sample_score = df['Recommended'].iloc[0]

print(sample)
print(analyzer.polarity_scores(sample))
print(sample_score)

pretty decent airline. moroni to moheli. turned out to be a pretty decent airline. online booking worked well, checkin and boarding was fine and the plane looked well maintained. its a very short flight just 20 minutes or so so i did not expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. both flights on time.
{'neg': 0.0, 'neu': 0.783, 'pos': 0.217, 'compound': 0.9342}
1


> ### **Sample Test Result**
>
> The VADER score aligns well with the review content and the target variable (`Recommended`), confirming that the sentiment extraction is working as expected. We can now apply this to the full dataset.

## **3. Apply VADER to All Reviews**

In [6]:
def get_vader_scores(text):
    scores = analyzer.polarity_scores(text)
    return pd.Series([scores['compound'], scores['pos'], scores['neg'], scores['neu']])

df[['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']] = df['cleaned_review_BE'].apply(get_vader_scores)

print("VADER scores extracted!")
df[['cleaned_review_BE', 'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']].head()

VADER scores extracted!


,cleaned_review_BE,vader_compound,vader_pos,vader_neg,vader_neu
0,pretty decent airline. moroni to moheli. turne...,0.9342,0.217,0.000,0.783
1,not a good airline. moroni to anjouan. it is a...,-0.8244,0.027,0.091,0.882
2,flight was fortunately short. anjouan to dzaou...,0.7569,0.106,0.027,0.867
3,i will never fly again with adria. please do a...,-0.9600,0.034,0.177,0.789
4,it ruined our last days of holidays. do not bo...,0.3027,0.103,0.087,0.810


In [ ]:
# Check alignment between VADER compound and Recommended
df['vader_predicted'] = (df['vader_compound'] > 0).astype(int)
mismatch = (df['vader_predicted'] != df['Recommended']).sum()
match_rate = (1 - mismatch / len(df)) * 100

print(f"Match rate: {match_rate:.1f}%")
print(f"Mismatched rows: {mismatch}")

Match rate: 81.8%
Mismatched rows: 4185


In [ ]:
# Sample mismatched reviews
mismatched = df[df['vader_predicted'] != df['Recommended']]

print(mismatched['cleaned_review_BE'].iloc[0])
print(mismatched[['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']].iloc[0])
print(mismatched['Recommended'].iloc[0])

flight was fortunately short. anjouan to dzaoudzi. a very small airline and the only airline based in comoros. check in was disorganised because of locals with big packages and disinterested staff. the flight was fortunately short 30 mins . took off on time and landed on time. with a short flight like there was of course no in flight entertainment nor cabin service except for biscuits and a bottle of water, which was quite nice!
vader_compound    0.7569
vader_pos         0.1060
vader_neg         0.0270
vader_neu         0.8670
Name: 2, dtype: float64
0


> ### **VADER vs. Actual Recommendation: Alignment Check**
>
> To assess how well document-level sentiment aligns with passenger recommendation decisions, the sign of `vader_compound` was compared against the actual `Recommended` label.
>
> - **Match rate: 81.8%** (4,185 mismatched reviews out of 22,980)
>
> An 81.8% alignment is reasonable, considering that `Recommended` reflects a passenger's overall judgment while VADER captures only the sentiment expressed in the review text.
>
>
> However, certain edge cases reveal its limitations. The mismatched cases often involve reviews with mixed sentiment (e.g., positive remarks about food or staff alongside a negative recommendation due to a flight delay), where document-level sentiment fails to capture which specific aspect drove the recommendation decision. 
>
> - **vader_compound: 0.7569** (strongly positive)
> - **Recommended:    No** (0)
>
> This limitation motivates the need for aspect-level sentiment analysis (Set C), which may better isolate the service dimensions that actually influence passenger recommendations.

## **4. Save Dataset**

In [18]:
# Drop unnecessary column
df = df.drop(columns=['vader_predicted'])
df.columns

Index(['Airline Name', 'Verified', 'Type Of Traveller', 'Seat Type',
       'Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
       'Ground Service', 'Inflight Entertainment', 'Wifi & Connectivity',
       'Value For Money', 'Recommended', 'Covid_Period', 'review_length',
       'Full_Review', 'cleaned_review_BE', 'cleaned_review_C',
       'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu'],
      dtype='object')

In [19]:
# Save VADER Scores
import os
os.makedirs('../1_data/processed', exist_ok=True)
out_path = '../1_data/processed/03_vader_scores.csv'
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Final shape: {df.shape}")
df.info()

Saved → ../1_data/processed/03_vader_scores.csv
Final shape: (22980, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22980 entries, 0 to 22979
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Airline Name            22980 non-null  object 
 1   Verified                22980 non-null  bool   
 2   Type Of Traveller       22980 non-null  object 
 3   Seat Type               22980 non-null  object 
 4   Seat Comfort            18763 non-null  float64
 5   Cabin Staff Service     18673 non-null  float64
 6   Food & Beverages        14162 non-null  float64
 7   Ground Service          18292 non-null  float64
 8   Inflight Entertainment  10245 non-null  float64
 9   Wifi & Connectivity     5896 non-null   float64
 10  Value For Money         21806 non-null  float64
 11  Recommended             22980 non-null  int64  
 12  Covid_Period            22980 non-null  int64  
 13  review_length     